# INFIDRUG MOLEQULAR FUSION — Variational Quantum Regression (VQR)
#
# Team TESSARACT. Target: measured log-solubility (mol/L) on delaney.csv.
# Shared pipeline: 256 SMILES features -> 4-component TruncatedSVD ->
# training-fitted scaling -> arctan angle map.

### Cell 1 — Imports

In [ ]:
code
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import roc_curve, roc_auc_score

### Cell 2 — Load dataset and group-aware outer/inner split

In [ ]:
code
# Load dataset
df = pd.read_csv("delaney.csv")
SMILES_COL = "SMILES"
TARGET_COL = "measured log(solubility:mol/L)"

smiles = df[SMILES_COL].astype(str)
y = df[TARGET_COL].astype(float).to_numpy()
groups = smiles.to_numpy()

# OUTER split -> final test set
outer = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
dev_idx, test_idx = next(outer.split(smiles, y, groups))

# INNER split -> training and validation
inner = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=43)
tr_rel, val_rel = next(inner.split(smiles.iloc[dev_idx], y[dev_idx], groups[dev_idx]))

train_idx = dev_idx[tr_rel]
val_idx = dev_idx[val_rel]

y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

### Cell 3 — Feature preparation (256 features -> 4 components -> arctan)

In [ ]:
code
# 1) SMILES -> maximum 256 character/ngram count features
vectorizer = CountVectorizer(
    analyzer="char",
    ngram_range=(1, 3),
    lowercase=False,
    min_df=2,
    max_features=256,
)
vectorizer.fit(smiles.iloc[train_idx])
X256 = vectorizer.transform(smiles).toarray().astype(float)

# 2) 256 features -> 4 SVD components
svd = TruncatedSVD(n_components=4, random_state=42)
svd.fit(X256[train_idx])
X4 = svd.transform(X256)

# 3) Scale using training data only
scaler = StandardScaler()
scaler.fit(X4[train_idx])
X4 = scaler.transform(X4)

# 4) Map numbers to bounded angles for quantum encoding
X = np.arctan(X4)

print("Original features:", X256.shape[1])
print("Quantum input features:", X.shape[1])
print("Variance retained:", round(svd.explained_variance_ratio_.sum() * 100, 2), "%")

### Cell 4 — Evaluation metrics and plotting function

In [ ]:
code
def evaluate(name, actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2 = r2_score(actual, predicted)

    print(name)
    print("MAE :", round(mae, 4))
    print("RMSE:", round(rmse, 4))
    print("R²  :", round(r2, 4))

    plt.figure(figsize=(6, 4))
    plt.scatter(actual, predicted, alpha=0.65)
    lo = min(actual.min(), predicted.min())
    hi = max(actual.max(), predicted.max())
    plt.plot([lo, hi], [lo, hi], "--")
    plt.xlabel("Actual logS")
    plt.ylabel("Predicted logS")
    plt.title(name + " - Actual vs Predicted")
    plt.show()

    return {"MAE": mae, "RMSE": rmse, "R2": r2}

### Cell 5 — Pure-Python 4-qubit simulator
#
# Gates: RY, RZ, CNOT. Encoding maps 4 features to 4 qubits; measurement
# returns one Pauli-Z expectation per qubit. quantum_similarity is the
# squared overlap between quantum states.

In [ ]:
code
# -------- Simple 4-qubit simulator --------
N_QUBITS = 4
DIM = 2 ** N_QUBITS
basis = np.arange(DIM)


def zero_state(n_samples):
    state = np.zeros((n_samples, DIM), dtype=complex)
    state[:, 0] = 1
    return state


def RY(state, angle, q):
    zero = basis[(basis & (1 << q)) == 0]
    one = zero | (1 << q)
    a = state[:, zero].copy()
    b = state[:, one].copy()
    c = np.cos(np.asarray(angle) / 2)
    s = np.sin(np.asarray(angle) / 2)
    if np.ndim(c) > 0:
        c, s = c[:, None], s[:, None]
    state[:, zero] = c * a - s * b
    state[:, one] = s * a + c * b
    return state


def RZ(state, angle, q):
    zero = basis[(basis & (1 << q)) == 0]
    one = zero | (1 << q)
    phase = np.exp(0.5j * np.asarray(angle))
    if np.ndim(phase) > 0:
        phase = phase[:, None]
    state[:, zero] *= np.conj(phase)
    state[:, one] *= phase
    return state


def CNOT(state, control, target):
    order = basis ^ (((basis >> control) & 1) << target)
    return state[:, order]


def encode(features, depth=2):
    """Encode 4 numerical features into 4 qubits."""
    state = zero_state(len(features))
    for _ in range(depth):
        for q in range(4):
            state = RY(state, features[:, q], q)  # encode feature
            state = RZ(state, features[:, q], q)
        for q in range(3):
            state = CNOT(state, q, q + 1)  # qubit interaction
    return state


def measure_Z(state):
    """Return one Z expectation value for each qubit."""
    probability = np.abs(state) ** 2
    signs = np.stack([1 - 2 * ((basis >> q) & 1) for q in range(4)], axis=1)
    return probability @ signs


def quantum_similarity(A, B):
    """Squared overlap between quantum states."""
    return np.abs(A.conj() @ B.T) ** 2

## Notebook 4 — Variational Quantum Regression (VQR)
#
# Architecture: 4 input features mapped into a parameterized quantum circuit
# with 16 variational rotation parameters theta, a Pauli-Z expectation on
# qubit 0, a linear scaling head (a * <Z0> + b), optimised with L-BFGS-B.

### Cell 6 — VQR training: trainable 16-parameter circuit + L-BFGS-B

In [ ]:
code
from scipy.optimize import minimize


def variational_output(features, theta):
    """Trainable 4-qubit circuit. Returns measurement of qubit 1."""
    state = zero_state(len(features))

    # Two circuit layers
    for layer in range(2):
        # Encode molecular features in the first layer
        if layer == 0:
            for q in range(4):
                state = RY(state, features[:, q], q)
                state = RZ(state, features[:, q], q)

        # Trainable rotations: theta values are changed by optimizer
        for q in range(4):
            i = layer * 8 + q * 2
            state = RY(state, theta[i], q)
            state = RZ(state, theta[i + 1], q)

        # Connect qubits
        for q in range(3):
            state = CNOT(state, q, q + 1)

    return measure_Z(state)[:, 0]


# 16 quantum parameters + 2 classical output parameters
rng = np.random.default_rng(42)
theta0 = rng.normal(0, 0.3, 16)

# Start the final linear output from a simple least-squares fit
initial_measurement = variational_output(X[train_idx], theta0)
a, b = np.polyfit(initial_measurement, y_train, 1)
initial_parameters = np.concatenate([theta0, [a, b]])


def predict_vqr(features, parameters):
    theta = parameters[:16]
    a, b = parameters[16], parameters[17]
    measurement = variational_output(features, theta)
    return a * measurement + b


def loss(parameters):
    prediction = predict_vqr(X[train_idx], parameters)
    return np.mean((y_train - prediction) ** 2)


history = []


def save_loss(parameters):
    history.append(loss(parameters))


# Train: repeatedly change theta to reduce error
result = minimize(
    loss,
    initial_parameters,
    method="L-BFGS-B",
    callback=save_loss,
    options={"maxiter": 160},
)

best_parameters = result.x

print("Training stopped after", result.nit, "iterations")
print("Final training loss:", round(result.fun, 4))

# Validation and test
val_prediction = predict_vqr(X[val_idx], best_parameters)
print("Validation RMSE:",
      round(np.sqrt(mean_squared_error(y_val, val_prediction)), 4))

test_prediction = predict_vqr(X[test_idx], best_parameters)
vqr_metrics = evaluate("VQR", y_test, test_prediction)

# Show how the error reduced during training
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(history) + 1), history)
plt.xlabel("Optimizer iteration")
plt.ylabel("Training MSE")
plt.title("VQR Training Loss")
plt.show()

### Cell 7 — Predict new molecules with the trained VQR circuit

In [ ]:
code
new_smiles = ["CCO", "c1ccccc1"]

new256 = vectorizer.transform(new_smiles).toarray().astype(float)
new4 = scaler.transform(svd.transform(new256))
newX = np.arctan(new4)

pd.DataFrame({
    "SMILES": new_smiles,
    "Predicted logS": predict_vqr(newX, best_parameters),
})

### Cell 8 — ROC analysis and save prediction vector

In [ ]:
code
# Extra ROC analysis: logS >= -2.0 is treated as positive
threshold = -2.0
y_binary = (y_test >= threshold).astype(int)

fpr, tpr, _ = roc_curve(y_binary, test_prediction)
auc_value = roc_auc_score(y_binary, test_prediction)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, linewidth=2, label=f"AUC = {auc_value:.4f}")
plt.plot([0, 1], [0, 1], "--", label="Random Guess (AUC = 0.5000)")
plt.xlabel("False Positive Rate (1 - Specificity)")
plt.ylabel("True Positive Rate (Sensitivity)")
plt.title("ROC Curve (Threshold: logS >= -2.0)")
plt.legend(loc="lower right")
plt.grid(alpha=0.25)
plt.show()

# VQR
np.save("vqr_pred.npy", test_prediction)
print("ROC-AUC:", round(auc_value, 4))